# Train Baseline Models — LSTM and GAT on all 5 datasets

Trains two baseline trajectory predictors **from scratch** on all 5 ETH/UCY datasets
to compare against Social-STGCNN:

- **`social_lstm`** — per-pedestrian encoder-decoder LSTM (no social interaction)
- **`social_gat`** — per-frame Graph Attention + temporal LSTM

A single run trains all **10 combinations** (2 models × 5 datasets).
**Loss:** NLL only — no collision loss, these are baselines.

## 1. Environment setup

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(),
      '-', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')


In [ ]:
%pip install -q networkx tqdm pandas


## 2. Mount Google Drive & set project directory

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# import os
# PROJECT_DIR = '/content/drive/MyDrive/DLproject-Social-STGCNN'
# assert os.path.isdir(PROJECT_DIR), f'Project not found at {PROJECT_DIR}'
# os.chdir(PROJECT_DIR)
# print('CWD:', os.getcwd())


In [ ]:
# Local machine
import os
PROJECT_DIR = r'D:\OMSCS\DLproject-Social-STGCNN'
assert os.path.isdir(PROJECT_DIR), f'Repo not found at {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())


## 3. Imports

In [ ]:
import os, copy, pickle, math
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from IPython.display import clear_output

from utils import TrajectoryDataset
from model_extras import build_model_from_args

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


## 4. Configuration

`MODEL_TYPES` and `DATASETS` are lists — the training cell iterates all combinations automatically.

In [ ]:
# ── What to train ──────────────────────────────────────────────────────────
MODEL_TYPES = ['lstm', 'gat']
DATASETS    = ['eth', 'hotel', 'univ', 'zara1', 'zara2']
NUM_EPOCHS  = 200   # reduce to 200 due to time-constrained

# ── Fixed training hyperparameters (match Social-STGCNN baseline) ─────────
INPUT_SIZE   = 2
OUTPUT_SIZE  = 5
OBS_SEQ_LEN  = 8
PRED_SEQ_LEN = 12
BATCH_SIZE   = 128
LR           = 0.01
LR_SH_RATE   = 150
USE_LRSCHD   = True
HIDDEN_DIM   = 64
NUM_LAYERS   = 1
DROPOUT      = 0.1

print(f'Will train {len(MODEL_TYPES) * len(DATASETS)} combos:')
for mt in MODEL_TYPES:
    for ds in DATASETS:
        tag  = f'social-{mt}-{ds}'
        ckpt = f'./baseline_{mt}/{tag}'
        done = '(exists)' if os.path.isdir(ckpt) else ''
        print(f'  {tag:30s}  ->  {ckpt}  {done}')

## 5. Data loader helper

In [ ]:
def build_loaders(dataset, obs_len=OBS_SEQ_LEN, pred_len=PRED_SEQ_LEN):
    root = f'./datasets/{dataset}/'
    dset_train = TrajectoryDataset(root + 'train/', obs_len=obs_len, pred_len=pred_len,
                                   skip=1, norm_lap_matr=True)
    dset_val   = TrajectoryDataset(root + 'val/',   obs_len=obs_len, pred_len=pred_len,
                                   skip=1, norm_lap_matr=True)
    loader_train = DataLoader(dset_train, batch_size=1, shuffle=True,  num_workers=0)
    loader_val   = DataLoader(dset_val,   batch_size=1, shuffle=False, num_workers=0)
    print(f'  [{dataset}] train={len(dset_train)} scenes  val={len(dset_val)} scenes')
    return loader_train, loader_val

print('build_loaders() defined.')

## 6. Loss function

In [ ]:
def bivariate_loss(V_pred, V_trgt):
    normx = V_trgt[:,:,0] - V_pred[:,:,0]
    normy = V_trgt[:,:,1] - V_pred[:,:,1]
    sx    = torch.exp(V_pred[:,:,2])
    sy    = torch.exp(V_pred[:,:,3])
    corr  = torch.tanh(V_pred[:,:,4])
    sxsy  = sx * sy
    z     = (normx/sx)**2 + (normy/sy)**2 - 2*corr*normx*normy/sxsy
    negRho = 1 - corr**2
    result = torch.exp(-z / (2*negRho))
    denom  = 2 * math.pi * sxsy * torch.sqrt(negRho)
    return -torch.log(torch.clamp(result/denom, min=1e-20)).mean()

print('bivariate_loss() defined.')

## 7. Training loop function

In [ ]:
PLOT_EVERY = 10

def _plot(metrics, best, epoch, total_epochs, tag, save_path=None):
    fig, ax = plt.subplots(figsize=(8, 4))
    ep = range(len(metrics['train_loss']))
    ax.plot(ep, metrics['train_loss'], label='Train NLL', color='steelblue', linewidth=1.6)
    ax.plot(ep, metrics['val_loss'],   label='Val NLL',   color='crimson',   linewidth=1.6, linestyle='--')
    if best['epoch'] >= 0:
        ax.axvline(best['epoch'], color='green', linestyle=':', linewidth=1.2,
                   alpha=0.8, label=f'val_best (ep {best["epoch"]})')
    ax.set_xlabel('Epoch'); ax.set_ylabel('NLL')
    ax.set_title(f'{tag}  —  ep {epoch+1}/{total_epochs}')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=110, bbox_inches='tight')
    plt.show()


def run_training(args, model, loader_train, loader_val, ckpt_dir, tag):
    """Train model for args.num_epochs; save checkpoints to ckpt_dir."""
    metrics    = {'train_loss': [], 'val_loss': []}
    best       = {'epoch': -1, 'val_loss': float('inf')}
    num_epochs = args.num_epochs

    optimizer = optim.SGD(model.parameters(), lr=args.lr)
    scheduler = (optim.lr_scheduler.StepLR(optimizer, step_size=args.lr_sh_rate, gamma=0.2)
                 if args.use_lrschd else None)

    loader_len = len(loader_train)
    turn_point = int(loader_len / BATCH_SIZE) * BATCH_SIZE + loader_len % BATCH_SIZE - 1

    for epoch in range(num_epochs):
        # ── Train ────────────────────────────────────────────────────────────
        model.train()
        loss_sum  = 0.0
        n_updates = batch_count = 0
        is_fst    = True

        for cnt, batch in enumerate(loader_train):
            batch_count += 1
            batch = [t.to(device) for t in batch]
            _, _, _, _, _, _, V_obs, A_obs, V_tr, _ = batch

            optimizer.zero_grad()
            V_pred, _ = model(V_obs.permute(0,3,1,2), A_obs.squeeze())
            V_pred = V_pred.permute(0,2,3,1).squeeze()
            l = bivariate_loss(V_pred, V_tr.squeeze())

            if not torch.isfinite(l):
                is_fst = True
                continue

            if batch_count % BATCH_SIZE != 0 and cnt != turn_point:
                if is_fst: loss_acc = l;          is_fst = False
                else:       loss_acc = loss_acc + l
            else:
                (loss_acc / BATCH_SIZE).backward()
                optimizer.step()
                loss_sum  += (loss_acc / BATCH_SIZE).item()
                n_updates += 1
                is_fst     = True

        metrics['train_loss'].append(loss_sum / max(n_updates, 1))

        # ── Validate ─────────────────────────────────────────────────────────
        model.eval()
        vl_sum    = 0.0
        v_updates = v_count = 0
        is_fst    = True
        vloader_len = len(loader_val)
        v_turn = int(vloader_len / BATCH_SIZE) * BATCH_SIZE + vloader_len % BATCH_SIZE - 1

        with torch.no_grad():
            for cnt, batch in enumerate(loader_val):
                v_count += 1
                batch = [t.to(device) for t in batch]
                _, _, _, _, _, _, V_obs, A_obs, V_tr, _ = batch
                V_pred, _ = model(V_obs.permute(0,3,1,2), A_obs.squeeze())
                V_pred = V_pred.permute(0,2,3,1).squeeze()
                l = bivariate_loss(V_pred, V_tr.squeeze())
                if v_count % BATCH_SIZE != 0 and cnt != v_turn:
                    if is_fst: vl_acc = l;          is_fst = False
                    else:       vl_acc = vl_acc + l
                else:
                    vl_sum    += (vl_acc / BATCH_SIZE).item()
                    v_updates += 1
                    is_fst     = True

        avg_val = vl_sum / max(v_updates, 1)
        metrics['val_loss'].append(avg_val)

        if avg_val < best['val_loss']:
            best['val_loss'] = avg_val
            best['epoch']    = epoch
            torch.save(model.state_dict(), os.path.join(ckpt_dir, 'val_best.pth'))

        torch.save(model.state_dict(), os.path.join(ckpt_dir, 'epoch_final.pth'))

        if scheduler is not None:
            scheduler.step()

        with open(os.path.join(ckpt_dir, 'metrics.pkl'),          'wb') as f: pickle.dump(metrics, f)
        with open(os.path.join(ckpt_dir, 'constant_metrics.pkl'), 'wb') as f: pickle.dump(best, f)

        is_last = (epoch == num_epochs - 1)
        if epoch % PLOT_EVERY == 0 or is_last:
            clear_output(wait=True)
            print(f'{tag}   ep {epoch+1:3d}/{num_epochs}   '
                  f'train={metrics["train_loss"][-1]:.5f}   '
                  f'val={avg_val:.5f}   '
                  f'best_ep={best["epoch"]}  best_val={best["val_loss"]:.5f}')
            save_path = os.path.join(ckpt_dir, 'training_curve.png') if is_last else None
            _plot(metrics, best, epoch, num_epochs, tag, save_path=save_path)

    print(f'Done [{tag}]  best_val={best["val_loss"]:.5f}  epoch={best["epoch"]}')
    return metrics, best

print('run_training() defined.')

## 8. Train all combinations

In [ ]:
total = len(MODEL_TYPES) * len(DATASETS)
done  = 0

for dataset in DATASETS:
    print(f'\n{"="*60}')
    print(f'Loading data for dataset: {dataset}')
    loader_train, loader_val = build_loaders(dataset)

    for model_type in MODEL_TYPES:
        done += 1
        tag      = f'social-{model_type}-{dataset}'
        save_dir = f'./baseline_{model_type}'
        ckpt_dir = os.path.join(save_dir, tag)
        os.makedirs(ckpt_dir, exist_ok=True)

        args = SimpleNamespace(
            model_type=model_type,
            input_size=INPUT_SIZE,   output_size=OUTPUT_SIZE,
            obs_seq_len=OBS_SEQ_LEN, pred_seq_len=PRED_SEQ_LEN,
            n_stgcnn=1, n_txpcnn=5,  kernel_size=3,
            hidden_dim=HIDDEN_DIM,   num_layers=NUM_LAYERS,  dropout=DROPOUT,
            dataset=dataset,         batch_size=BATCH_SIZE,  num_epochs=NUM_EPOCHS,
            lr=LR,                   lr_sh_rate=LR_SH_RATE,  use_lrschd=USE_LRSCHD,
            tag=tag,                 save_dir=save_dir,
            collision_loss='none',   lambda_col=0.0,         d_min=0.2,
            ecp_k=10,                clip_grad=None,
        )
        with open(os.path.join(ckpt_dir, 'args.pkl'), 'wb') as f:
            pickle.dump(args, f)

        model = build_model_from_args(args).to(device)
        n_params = sum(p.numel() for p in model.parameters())
        print(f'\n[{done}/{total}]  Training {tag}  ({n_params:,} params)')

        run_training(args, model, loader_train, loader_val, ckpt_dir, tag)

print(f'\n{"="*60}')
print('All baselines trained.')

## 9. Verify outputs

In [ ]:
print('Checkpoint summary:')
for model_type in MODEL_TYPES:
    for dataset in DATASETS:
        tag      = f'social-{model_type}-{dataset}'
        ckpt_dir = os.path.join(f'./baseline_{model_type}', tag)
        if os.path.isdir(ckpt_dir):
            files = sorted(os.listdir(ckpt_dir))
            sizes = [f'{fn}: {os.path.getsize(os.path.join(ckpt_dir,fn))/1024:.1f}KB' for fn in files]
            print(f'  {tag:30s}  {", ".join(sizes)}')
        else:
            print(f'  {tag:30s}  NOT FOUND')